# Recommend — Evaluation (temporal leave-one-out + beyond-accuracy + ablation)

Third notebook in the series, separate from `Recommend_V3.ipynb` and
`Recommend_Model_Comparison.ipynb`. Scope decided after the model
comparison run (2026-09-25): since language condition showed no detectable
aggregate effect, this evaluation runs **`translated_en` only, across all
three models** (`tfidf`, `bge-m3`, `qwen3-embedding-0.6b`) — half the compute
of a full 2×3 grid, for the actual apples-to-apples answer the model
comparison's raw similarity scores couldn't give (similarity scale differs
by model — 0.61 mean for bge-m3 vs 0.51 for Qwen3 reflects each model's own
geometry, not which recommends better).

## What this notebook computes

1. **Temporal leave-one-out** — Precision@3, Recall@3, MRR, against a
   random baseline (analytic) and a popularity baseline (empirical).
2. **Ablation** — the same metrics with each re-ranking boost zeroed out
   one at a time, reusing the same embeddings (no re-embedding needed).
3. **Small multilingual sanity check** — hit@3 rate split by whether the
   held-out course's language matches the employee's (recomputed)
   inferred language.
4. **Beyond-accuracy metrics** (coverage, novelty, diversity) — computed
   directly from the *existing* `translated_en` recommendation CSVs
   `Recommend_Model_Comparison.ipynb` already produced. No re-embedding at
   all for this part — pure pandas plus one small TF-IDF similarity
   computation for diversity, so it's the cheapest section to run and
   needs no GPU.

## Design decisions, flagged rather than silently assumed

- **Eligible-employee set**: `Skills_Source == "real"` employees (their
  `Skills` field was built entirely from actual completed history, not
  simulated) with **2+ qualifying completions**
  (`Status == "completed" AND matched_catalog == True` in
  `history_cleaned.csv` — the exact filter `build_final_skills.py` used to
  decide `Skills_Source`). This matches `WHATS_LEFT.md` §3's stated scope
  (173 real-skills employees, further reduced to 2+ completions).
- **Held-out item = the most recent qualifying completion** (by
  `Terminé à (PST/PDT)`), not a randomly sampled one — this is the
  *temporal* leave-one-out variant, which avoids leaking a user's future
  behavior into the evaluation (see `PROJECT_JOURNAL.md` Phase 21).
  Employees where every qualifying row has a missing/unparseable
  completion date are **dropped from eligibility and counted**, not
  silently defaulted to a proxy date — an honesty call, not a fallback.
- **Held-out course must exist in the active candidate pool** — a
  completed course that has since gone retired, or is `Level == "general"`,
  cannot structurally be recommended regardless of the model. Employees
  whose held-out course fails this check are dropped and counted
  separately (`unscoreable`), not scored as an automatic miss.
- **Re-deriving the employee, not just re-scoring** — unlike V3 and the
  model comparison notebook (which only re-derive `Excluded_Course_IDs`
  and `Inferred_Language` at runtime), leave-one-out here also has to
  **regenerate `Skills`** for each eligible employee with the held-out
  completion removed — otherwise the held-out course's own skill tags
  would still leak into the employee's embedding text even though the
  "completion" itself is hidden. Uses the identical real-skills logic from
  `build_final_skills.py` §2 (qualifying completions → union of
  `Skills_EN` from the catalog), just re-run per employee minus one row.
- **MRR rank convention**: rank = 1 + (number of *eligible* courses scoring
  strictly higher than the held-out course). Courses tied with the
  held-out course's score are not counted against it — documented since
  ties are realistic, especially for TF-IDF where many courses can share
  a score of exactly 0.
- **Random baseline is analytic, not simulated** — for a uniformly random
  ranking among `N` eligible courses, `E[hit@3] = min(3,N)/N` and
  `E[MRR] = H_N / N` (`H_N` = the `N`-th harmonic number). Exact and
  deterministic rather than averaged over simulated shuffles.
- **Popularity baseline** ranks courses by global completion count in
  `history_cleaned.csv`, **corrected per employee** by subtracting that
  employee's own held-out row from the count (avoids a small self-leak).
  Ties (very common — most of the ~10,765-course pool has zero recorded
  completions) are broken with a small fixed-seed random jitter, not
  catalog row order, to avoid a systematic bias toward whichever courses
  happen to sort first.
- **Ablation reuses cached embeddings** — only the boost weights change
  (one zeroed at a time: field, preference, language, level, training-type,
  domain-gate), so no model is re-run. Fast once the base embeddings for a
  model are computed.

## Not run in this notebook

Same constraint as the other two notebooks — no GPU/network access here
for `bge-m3` / `Qwen3-Embedding-0.6B`. Sections 1-11 (leave-one-out +
ablation) need a local run. **Section 12 (beyond-accuracy) does not need
GPU or network at all** — it only reads the three `recommendations_translated_en_*.csv`
files `Recommend_Model_Comparison.ipynb` already saved, so it's runnable
immediately, independent of the rest.


## 1. Setup & config

In [ ]:
import re
import time
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

BASE = Path.cwd()
if not (BASE / "data" / "processed").is_dir():
    BASE = BASE.parent

DATA_PROCESSED = BASE / "data" / "processed"
COMPARISON_OUTPUTS = BASE / "outputs" / "model_comparison"   # from Recommend_Model_Comparison.ipynb
OUTPUTS = BASE / "outputs" / "evaluation"
OUTPUTS.mkdir(parents=True, exist_ok=True)

EMPLOYEES_PATH = DATA_PROCESSED / "employees_final.csv"
CATALOG_FULL_PATH = DATA_PROCESSED / "catalog_full_en_fr_translated.csv"
HISTORY_PATH = DATA_PROCESSED / "history_cleaned.csv"

RANDOM_SEED = 42
TOP_K = 3
MODELS = ["tfidf", "bge-m3", "qwen3-embedding-0.6b"]


def to_plain_array(series: pd.Series) -> np.ndarray:
    '''Force a plain numpy object array out of a pandas Series -- see
    Recommend_Model_Comparison.ipynb's intro for why this is needed on
    pandas 3.x (StringArray doesn't support the [:, None] broadcasting
    the language/level comparisons below rely on).'''
    return series.astype(object).to_numpy()


## 2. Load the three real files

In [ ]:
employees_all = pd.read_csv(EMPLOYEES_PATH).fillna("")
catalog_full = pd.read_csv(CATALOG_FULL_PATH).fillna("")
history = pd.read_csv(HISTORY_PATH).fillna("")

assert employees_all["Email"].is_unique, "Employee emails should be unique"
print("employees_all:", employees_all.shape)
print("catalog_full:", catalog_full.shape)
print("history:", history.shape)
print("Skills_Source breakdown:")
print(employees_all["Skills_Source"].value_counts())


## 3. Derive the active-only candidate pool (translated_en text only)

Identical filters to V3 / the model comparison notebook. This evaluation
only uses the `translated_en` text condition, per the scope decision above.

In [ ]:
n_before = len(catalog_full)
catalog_pool = catalog_full[
    (catalog_full["Status"].str.lower() == "active")
    & (catalog_full["Level"].str.lower() != "general")
].copy().reset_index(drop=True)

catalog_pool["_level_norm"] = catalog_pool["Level"].str.lower().str.strip()
catalog_pool["_lang"] = catalog_pool["Library"].str.upper().map({"EN-US": "EN", "FR-FR": "FR"}).fillna("")
catalog_pool["_text"] = (
    catalog_pool["Course Title_EN"].astype(str) + ". "
    + catalog_pool["Subjects_EN"].astype(str) + ". "
    + catalog_pool["Skills_EN"].astype(str) + ". "
    + catalog_pool["Description_EN"].astype(str)
).str.strip()

course_id_index = {str(cid): idx for idx, cid in enumerate(catalog_pool["Course ID"])}
n_course = len(catalog_pool)
print(f"Full catalog: {n_before} -> candidate pool: {n_course} rows")


## 4. Real-skills-from-history function (ported from `build_final_skills.py` §2)

Needed to **regenerate** `Skills` for each eligible employee with the
held-out completion removed — `employees_final.csv`'s existing `Skills`
column already bakes in the held-out course's skill tags for `real`
employees, which would leak information leave-one-out is supposed to hide.
Same qualifying filter (`completed` + `matched_catalog == True`) and the
same `Skills_EN`-union-by-email logic as the original script — just made
callable per employee, with one row optionally excluded.

In [ ]:
cat_lookup = catalog_full.set_index(catalog_full["Course ID"].astype(str))["Skills_EN"]

history["_course_id"] = history["ID de contenu"].astype(str)
history["_status_norm"] = history["Status"].str.strip().str.lower()
history["_completed_at"] = pd.to_datetime(history["Terminé à (PST/PDT)"], errors="coerce")

qualifying_mask = (history["_status_norm"] == "completed") & (history["matched_catalog"] == True)
qualifying_history = history[qualifying_mask].copy()

print(f"Qualifying (completed + matched_catalog) history rows: {len(qualifying_history)} / {len(history)}")
print(f"Qualifying rows with an unparseable completion date: {qualifying_history['_completed_at'].isna().sum()}")


def skills_from_courses(course_ids) -> str:
    skills = set()
    for cid in course_ids:
        skills_str = cat_lookup.get(cid, "")
        if pd.notna(skills_str) and skills_str:
            skills.update(s.strip() for s in str(skills_str).split(",") if s.strip())
    return ", ".join(sorted(skills))


## 5. Identify eligible employees + pick the held-out course (temporal LOO)

Three ways an otherwise-`real`, otherwise-2+-completions employee gets
dropped, each counted separately rather than silently merged into one
"ineligible" bucket:

1. Fewer than 2 qualifying completions.
2. Every qualifying completion has an unparseable/missing completion date
   (can't determine "most recent" without one).
3. The most-recent qualifying course isn't in the active candidate pool
   (retired, or now `Level == "general"`) — structurally unrecommendable
   regardless of model, so scoring it would be meaningless.

In [ ]:
real_employees = employees_all[employees_all["Skills_Source"] == "real"].copy()
print(f"Skills_Source == 'real' employees: {len(real_employees)}")

eligible_rows = []
dropped_too_few = 0
dropped_no_date = 0
dropped_not_in_pool = 0

for email, group in qualifying_history.groupby("email_n"):
    if email not in set(real_employees["Email"]):
        continue  # qualifying history for an employee not marked 'real' shouldn't happen, but guard anyway
    if len(group) < 2:
        dropped_too_few += 1
        continue

    dated = group.dropna(subset=["_completed_at"])
    if dated.empty:
        dropped_no_date += 1
        continue

    most_recent_row = dated.loc[dated["_completed_at"].idxmax()]
    held_out_id = most_recent_row["_course_id"]

    if held_out_id not in course_id_index:
        dropped_not_in_pool += 1
        continue

    remaining_ids = [cid for cid in group["_course_id"] if cid != held_out_id]
    eligible_rows.append({
        "Email": email,
        "held_out_course_id": held_out_id,
        "remaining_qualifying_ids": remaining_ids,
    })

eligible_df = pd.DataFrame(eligible_rows)
print(f"Eligible for leave-one-out: {len(eligible_df)}")
print(f"  dropped (fewer than 2 qualifying completions): {dropped_too_few}")
print(f"  dropped (no parseable completion date): {dropped_no_date}")
print(f"  dropped (held-out course not in active candidate pool): {dropped_not_in_pool}")


## 6. Build `employees_eval` — the eligible subset, with `Skills`/`Excluded_Course_IDs`/`Inferred_Language` regenerated

Only these rows are touched. Every other column (`Field_EN`, `Bu_new`,
`Employee_Level`, `Preferred_Formation`, `Preferred_Training_Type`) is
carried over unchanged from `employees_final.csv`.

`Excluded_Course_IDs` is rebuilt from the **full** completed+in-progress
history (V3's broader definition, not just `matched_catalog` rows) minus
the held-out course specifically — so every other course the employee has
engaged with stays excluded as normal, and only the held-out course is
added back as a recommendable candidate.

In [ ]:
LANGUAGE_THRESHOLD = 0.6
id_to_library = dict(zip(catalog_full["Course ID"].astype(str), catalog_full["Library"]))

eval_rows = []
for _, row in eligible_df.iterrows():
    email = row["Email"]
    held_out_id = row["held_out_course_id"]

    emp_row = real_employees[real_employees["Email"] == email].iloc[0].to_dict()

    # Regenerate Skills from remaining qualifying completions only.
    emp_row["Skills"] = skills_from_courses(row["remaining_qualifying_ids"])

    # Regenerate Excluded_Course_IDs: full completed+in-progress history for this
    # employee, minus the held-out course (V3's broader definition, not just
    # matched_catalog rows -- an employee could have unmatched-but-real engagement
    # with other courses too, and those should stay excluded).
    emp_hist = history[history["email_n"] == email]
    completed = emp_hist[emp_hist["_status_norm"] == "completed"]
    in_progress = emp_hist[emp_hist["_status_norm"] == "in-progress"]
    excluded_ids = set(completed["_course_id"]).union(set(in_progress["_course_id"]))
    excluded_ids.discard(held_out_id)
    emp_row["Excluded_Course_IDs"] = excluded_ids

    # Regenerate Inferred_Language from completed rows minus the held-out course.
    completed_minus_held_out = completed[completed["_course_id"] != held_out_id]
    langs = completed_minus_held_out["_course_id"].map(id_to_library).dropna()
    inferred_language = ""
    if len(langs) > 0:
        counts = Counter(langs)
        top_lang, top_count = counts.most_common(1)[0]
        if top_count / len(langs) > LANGUAGE_THRESHOLD:
            inferred_language = "EN" if top_lang == "en-US" else "FR" if top_lang == "fr-FR" else ""
    emp_row["Inferred_Language"] = inferred_language

    emp_row["_held_out_course_id"] = held_out_id
    emp_row["_held_out_library"] = id_to_library.get(held_out_id, "")
    eval_rows.append(emp_row)

employees_eval = pd.DataFrame(eval_rows).reset_index(drop=True)

employees_eval["_text"] = (
    employees_eval["Field_EN"].astype(str) + ". "
    + employees_eval["Skills"].astype(str) + ". "
    + employees_eval["Preferred_Formation"].astype(str)
).str.strip()

n_eval = len(employees_eval)
print(f"employees_eval: {n_eval} rows")
employees_eval[["Email", "Skills", "_held_out_course_id", "Inferred_Language"]].head()


## 7. Preference vocabulary gate (identical mechanism to the other two notebooks)

In [ ]:
PREFERENCE_VOCAB_STOPWORDS = {"and", "of", "the", "for", "with", "your", "from", "using", "into"}
PREFERENCE_VOCAB_BLOCKLIST = {"leadership skills", "professional development"}

def build_preference_vocab(catalog: pd.DataFrame) -> set:
    phrases = set()
    for col in ["Skills_EN", "Subjects_EN"]:
        for val in catalog[col].dropna():
            for term in str(val).split(","):
                term = term.strip().lower()
                if len(term.split()) >= 2 and term not in PREFERENCE_VOCAB_BLOCKLIST:
                    phrases.add(term)
    return phrases

PREFERENCE_VOCAB = build_preference_vocab(catalog_pool)

def preference_is_informative(text: str) -> bool:
    cleaned = str(text).strip().lower()
    if cleaned == "":
        return False
    return any(phrase in cleaned for phrase in PREFERENCE_VOCAB)

has_pref = employees_eval["Preferred_Formation"].apply(preference_is_informative).values
print(f"Eligible employees with an informative preference: {has_pref.sum()} / {n_eval}")


## 8. Field match — TF-IDF-weighted token overlap (identical mechanism)

In [ ]:
FIELD_STOPWORDS = {"and", "of", "the", "for", "role", "focus"}
MIN_FIELD_TOKEN_LEN = 4

def strip_accents(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(c))

def field_tokens(bu_new: str) -> list:
    raw = re.split(r"[^a-zA-Z]+", strip_accents(bu_new).lower())
    return [t for t in raw if len(t) >= MIN_FIELD_TOKEN_LEN and t not in FIELD_STOPWORDS]

unique_bu = employees_eval["Bu_new"].unique()
bu_tokens = {bu: set(field_tokens(bu)) for bu in unique_bu}
all_tokens = sorted(set().union(*bu_tokens.values())) if len(bu_tokens) else []
token_index = {t: i for i, t in enumerate(all_tokens)}

course_text_norm = catalog_pool["_text"].map(lambda t: strip_accents(t).lower())
course_token_matrix = np.zeros((n_course, len(all_tokens)), dtype=bool)
for tok, col in token_index.items():
    pattern = re.compile(r"\b" + re.escape(tok) + r"\b")
    course_token_matrix[:, col] = course_text_norm.map(lambda t: bool(pattern.search(t))).values

doc_freq = course_token_matrix.sum(axis=0)
token_idf = np.log((n_course + 1) / (doc_freq + 1)) + 1

emp_token_weights = np.zeros((n_eval, len(all_tokens)), dtype=np.float32)
for i, bu in enumerate(to_plain_array(employees_eval["Bu_new"])):
    for tok in bu_tokens[bu]:
        emp_token_weights[i, token_index[tok]] = token_idf[token_index[tok]]

raw_field_score = emp_token_weights @ course_token_matrix.astype(np.float32).T
emp_max_score = emp_token_weights.sum(axis=1, keepdims=True)
emp_max_score_safe = np.where(emp_max_score == 0, 1.0, emp_max_score)
field_score_norm = np.clip(raw_field_score / emp_max_score_safe, 0.0, 1.0)
print(f"field boost fires for {(field_score_norm > 0).mean():.1%} of (eligible employee, course) pairs")


## 9. Boost weights, embedding backends, and the flexible ranking function

`compute_rank_score_matrix` is the ablation-ready version of V3's scoring
step: it takes a dict of weight overrides (default = V3's unchanged
constants) and returns the full masked `rank_score` matrix — no
`argpartition`/top-3 selection, since leave-one-out only needs the rank of
one specific (known) course per employee, not a top-3 list.

In [ ]:
DEFAULT_WEIGHTS = {
    "FIELD_BOOST": 0.03,
    "PREFERENCE_BOOST": 0.15,
    "LANGUAGE_MATCH_BOOST": 0.07,
    "LANGUAGE_DEFAULT_EN_BOOST": 0.03,
    "LEVEL_MATCH_BOOST": 0.05,
    "LEVEL_MISMATCH_PENALTY": 0.10,
    "TRAINING_TYPE_BOOST": 0.05,
    "DOMAIN_GATE_PENALTY": 0.08,
}

ABLATIONS = {
    "all_boosts": {},
    "no_field": {"FIELD_BOOST": 0.0},
    "no_preference": {"PREFERENCE_BOOST": 0.0},
    "no_language": {"LANGUAGE_MATCH_BOOST": 0.0, "LANGUAGE_DEFAULT_EN_BOOST": 0.0},
    "no_level": {"LEVEL_MATCH_BOOST": 0.0, "LEVEL_MISMATCH_PENALTY": 0.0},
    "no_training_type": {"TRAINING_TYPE_BOOST": 0.0},
    "no_domain_gate": {"DOMAIN_GATE_PENALTY": 0.0},
}


def embed_tfidf(emp_texts, course_texts, pref_texts, has_pref):
    vectorizer = TfidfVectorizer()
    vectorizer.fit(list(emp_texts) + list(course_texts))
    emp_tfidf = vectorizer.transform(emp_texts)
    course_tfidf = vectorizer.transform(course_texts)
    sim_matrix = cosine_similarity(emp_tfidf, course_tfidf).astype(np.float32)
    pref_texts_safe = [t if h else "" for t, h in zip(pref_texts, has_pref)]
    pref_tfidf = vectorizer.transform(pref_texts_safe)
    pref_sim_matrix = cosine_similarity(pref_tfidf, course_tfidf).astype(np.float32)
    return sim_matrix, pref_sim_matrix


def embed_sentence_transformer(model_name, emp_texts, course_texts, pref_texts, has_pref, query_prompt_name=None):
    import torch
    from sentence_transformers import SentenceTransformer, util

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  torch.cuda.is_available(): {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")

    model = SentenceTransformer(model_name, device=device)
    emp_kwargs = {"prompt_name": query_prompt_name} if query_prompt_name else {}
    emp_vectors = model.encode(list(emp_texts), batch_size=64, normalize_embeddings=True,
                                show_progress_bar=True, **emp_kwargs)
    course_vectors = model.encode(list(course_texts), batch_size=64, normalize_embeddings=True,
                                   show_progress_bar=True)

    pref_vectors = np.zeros((len(emp_texts), emp_vectors.shape[1]), dtype=np.float32)
    has_pref = np.asarray(has_pref)
    if has_pref.any():
        pref_subset = [t for t, h in zip(pref_texts, has_pref) if h]
        pref_kwargs = {"prompt_name": query_prompt_name} if query_prompt_name else {}
        encoded = model.encode(pref_subset, batch_size=64, normalize_embeddings=True,
                                show_progress_bar=True, **pref_kwargs)
        pref_vectors[has_pref] = encoded

    sim_matrix = util.cos_sim(emp_vectors, course_vectors).numpy()
    pref_sim_matrix = util.cos_sim(pref_vectors, course_vectors).numpy()
    return sim_matrix, pref_sim_matrix


EMBEDDERS = {
    "tfidf": lambda emp_t, course_t, pref_t, has_pref: embed_tfidf(emp_t, course_t, pref_t, has_pref),
    "bge-m3": lambda emp_t, course_t, pref_t, has_pref: embed_sentence_transformer(
        "BAAI/bge-m3", emp_t, course_t, pref_t, has_pref, query_prompt_name=None,
    ),
    "qwen3-embedding-0.6b": lambda emp_t, course_t, pref_t, has_pref: embed_sentence_transformer(
        "Qwen/Qwen3-Embedding-0.6B", emp_t, course_t, pref_t, has_pref, query_prompt_name="query",
    ),
}


def compute_rank_score_matrix(sim_matrix, pref_sim_matrix, has_pref, field_score_norm, weight_overrides=None):
    w = dict(DEFAULT_WEIGHTS)
    if weight_overrides:
        w.update(weight_overrides)

    row_min = pref_sim_matrix.min(axis=1, keepdims=True)
    row_max = pref_sim_matrix.max(axis=1, keepdims=True)
    row_range_safe = np.where((row_max - row_min) == 0, 1.0, row_max - row_min)
    pref_norm = (pref_sim_matrix - row_min) / row_range_safe
    pref_contribution = np.where(has_pref[:, None], w["PREFERENCE_BOOST"] * pref_norm, 0.0)

    emp_lang = to_plain_array(employees_eval["Inferred_Language"])
    course_lang = to_plain_array(catalog_pool["_lang"])
    has_lang_pref = (emp_lang != "")[:, None]
    lang_match = has_lang_pref & (emp_lang[:, None] == course_lang[None, :])
    lang_default = (~has_lang_pref) & (course_lang[None, :] == "EN")
    lang_boost = np.where(lang_match, w["LANGUAGE_MATCH_BOOST"],
                           np.where(lang_default, w["LANGUAGE_DEFAULT_EN_BOOST"], 0.0))

    emp_level = to_plain_array(employees_eval["Employee_Level"])
    course_level = to_plain_array(catalog_pool["_level_norm"])
    course_has_level = (course_level != "")[None, :]
    level_match = course_has_level & (course_level[None, :] == emp_level[:, None])
    level_mismatch = course_has_level & (course_level[None, :] != emp_level[:, None])
    level_boost = np.where(level_match, w["LEVEL_MATCH_BOOST"],
                            np.where(level_mismatch, -w["LEVEL_MISMATCH_PENALTY"], 0.0))

    pref_type = to_plain_array(employees_eval["Preferred_Training_Type"].astype(str).str.strip())
    training_boost = np.where((pref_type == "Online")[:, None], w["TRAINING_TYPE_BOOST"], 0.0)

    field_boost = w["FIELD_BOOST"] * field_score_norm

    no_domain_signal = (field_score_norm == 0) & (pref_contribution == 0)
    domain_gate_boost = np.where(no_domain_signal, -w["DOMAIN_GATE_PENALTY"], 0.0)

    rank_score_matrix = (sim_matrix + field_boost + pref_contribution + lang_boost
                          + level_boost + training_boost + domain_gate_boost)

    exclusion_mask = np.zeros((n_eval, n_course), dtype=bool)
    for i, excluded in enumerate(employees_eval["Excluded_Course_IDs"]):
        for cid in excluded:
            col = course_id_index.get(cid)
            if col is not None:
                exclusion_mask[i, col] = True

    return np.where(exclusion_mask, -np.inf, rank_score_matrix)


## 10. Metrics helpers — leave-one-out rank, random baseline, popularity baseline

In [ ]:
def rank_of_held_out(rank_score_matrix_masked: np.ndarray) -> np.ndarray:
    '''For each eligible employee row, the rank (1-indexed) of their held-out
    course among all non-excluded candidates. Ties are NOT counted against
    the held-out course (see design note in the intro).'''
    held_out_cols = np.array([course_id_index[cid] for cid in employees_eval["_held_out_course_id"]])
    held_out_scores = rank_score_matrix_masked[np.arange(n_eval), held_out_cols]
    higher = (rank_score_matrix_masked > held_out_scores[:, None]).sum(axis=1)
    return higher + 1


def precision_recall_mrr(ranks: np.ndarray, k: int = TOP_K) -> dict:
    hit = ranks <= k
    return {
        "precision_at_k": float((hit.astype(float) / k).mean()),
        "recall_at_k": float(hit.mean()),
        "mrr": float((1.0 / ranks).mean()),
        "n": int(len(ranks)),
    }


# --- Analytic random baseline ---
n_eligible_per_employee = np.array([
    n_course - len(excluded) for excluded in employees_eval["Excluded_Course_IDs"]
])
max_n = n_eligible_per_employee.max()
harmonic_cumsum = np.cumsum(1.0 / np.arange(1, max_n + 1))  # harmonic_cumsum[N-1] == H_N

def random_baseline_metrics(k: int = TOP_K) -> dict:
    N = n_eligible_per_employee
    hit_rate = np.minimum(k, N) / N
    mrr = harmonic_cumsum[N - 1] / N
    return {
        "precision_at_k": float((hit_rate / k).mean()),
        "recall_at_k": float(hit_rate.mean()),
        "mrr": float(mrr.mean()),
        "n": int(len(N)),
    }


# --- Popularity baseline (empirical, per-employee corrected, tie-broken with fixed jitter) ---
global_popularity = history["_course_id"].value_counts()
popularity_by_col = np.zeros(n_course, dtype=np.float64)
for cid, idx in course_id_index.items():
    popularity_by_col[idx] = global_popularity.get(cid, 0)

_rng = np.random.default_rng(RANDOM_SEED)
jitter = _rng.uniform(0, 1e-6, size=n_course)

def popularity_baseline_metrics(k: int = TOP_K) -> dict:
    pop_matrix = np.tile(popularity_by_col + jitter, (n_eval, 1))
    # Per-employee correction: subtract this employee's own held-out row from its course's count.
    held_out_cols = np.array([course_id_index[cid] for cid in employees_eval["_held_out_course_id"]])
    pop_matrix[np.arange(n_eval), held_out_cols] -= 1.0
    # Apply the same exclusion mask as the real model (other completed/in-progress courses stay excluded).
    exclusion_mask = np.zeros((n_eval, n_course), dtype=bool)
    for i, excluded in enumerate(employees_eval["Excluded_Course_IDs"]):
        for cid in excluded:
            col = course_id_index.get(cid)
            if col is not None:
                exclusion_mask[i, col] = True
    pop_matrix_masked = np.where(exclusion_mask, -np.inf, pop_matrix)
    ranks = rank_of_held_out(pop_matrix_masked)
    return precision_recall_mrr(ranks, k=k)


## 11. Diagnostic — how popular are the 58 held-out courses themselves?

Before touching any weight or boost: the leave-one-out numbers above show
every content-based model losing badly to the popularity baseline. That
result has two very different possible explanations, and this diagnostic
is the cheap way to tell them apart — **it needs nothing computed in
section 12 below (no embeddings), so it can run right after section 10.**

- **If the held-out (most-recent-completion) courses are themselves
  unusually popular company-wide**, that's evidence "most recent
  completion" is, for a lot of employees, more a proxy for
  generic/assigned training than for an individually-relevant choice — a
  content-based system optimizing for topical fit isn't necessarily wrong
  just because it doesn't predict *that*.
- **If the held-out courses are spread roughly like a random draw from the
  catalog** (no more concentrated in popular courses than chance would
  produce), that points the other way — the content-based approach's core
  premise (topical similarity predicts what people actually complete)
  isn't well supported by this data, which is the more serious finding.

Compares the **observed** distribution of held-out courses' popularity
percentile (1 = most popular course in the active pool, 0 = least
popular) against what a **uniformly random** draw from the catalog would
produce, at several thresholds.

In [ ]:
# popularity_by_col and course_id_index come from section 10 above -- no re-embedding needed.
popularity_percentile_arr = pd.Series(popularity_by_col).rank(ascending=False, method="average").values / n_course
percentile_by_course_id = dict(zip(catalog_pool["Course ID"].astype(str), popularity_percentile_arr))

held_out_ids = employees_eval["_held_out_course_id"].astype(str)
held_out_percentiles = held_out_ids.map(percentile_by_course_id)

def pct_within(threshold: float) -> float:
    return float((held_out_percentiles <= threshold).mean())

most_popular_course_id = str(catalog_pool.loc[int(np.argmax(popularity_by_col)), "Course ID"])

diagnostic_summary = pd.DataFrame([
    {"metric": "n_eligible", "observed": len(held_out_ids), "expected_if_random": None},
    {"metric": "n_distinct_held_out_courses", "observed": held_out_ids.nunique(), "expected_if_random": None},
    {"metric": "median_popularity_percentile", "observed": round(float(held_out_percentiles.median()), 4), "expected_if_random": 0.50},
    {"metric": "pct_in_top_1pct_most_popular", "observed": round(pct_within(0.01), 4), "expected_if_random": 0.01},
    {"metric": "pct_in_top_5pct_most_popular", "observed": round(pct_within(0.05), 4), "expected_if_random": 0.05},
    {"metric": "pct_in_top_10pct_most_popular", "observed": round(pct_within(0.10), 4), "expected_if_random": 0.10},
    {"metric": "pct_in_top_25pct_most_popular", "observed": round(pct_within(0.25), 4), "expected_if_random": 0.25},
    {"metric": "pct_held_out_is_THE_single_most_popular_course", "observed": round(float((held_out_ids == most_popular_course_id).mean()), 4), "expected_if_random": round(1.0 / n_course, 6)},
])
print(diagnostic_summary.to_string(index=False))

diagnostic_path = OUTPUTS / "held_out_popularity_diagnostic.csv"
diagnostic_summary.to_csv(diagnostic_path, index=False)
print(f"\nSaved {diagnostic_path}")

# Which specific courses are most frequently the held-out target -- generic/broad-appeal
# courses recurring here would support the first explanation above; a long tail of
# distinct, role-specific courses would support the second.
title_lookup = catalog_pool.set_index(catalog_pool["Course ID"].astype(str))["Course Title_EN"]
held_out_counts = held_out_ids.value_counts().rename_axis("Course_ID").reset_index(name="times_held_out")
held_out_counts["Course_Title"] = held_out_counts["Course_ID"].map(title_lookup)
held_out_counts["popularity_percentile"] = held_out_counts["Course_ID"].map(percentile_by_course_id).round(3)

held_out_freq_path = OUTPUTS / "held_out_course_frequency.csv"
held_out_counts.to_csv(held_out_freq_path, index=False)
print(f"Saved {held_out_freq_path}\n")
print(held_out_counts.head(15).to_string(index=False))

## 12. Run leave-one-out + ablation for all three models

Each model is embedded **once** (course pool + eligible-employee subset,
~a tenth of the full 1,059-employee grid's employee side, so the dominant
cost — embedding ~10,765 courses — is the same as before, but the employee
side is much cheaper). Ablation then reuses the same `sim_matrix`/
`pref_sim_matrix` for all 7 weight configurations (all-boosts baseline + 6
single-boost-zeroed variants) — no re-embedding per ablation row.

In [ ]:
emp_texts = employees_eval["_text"].tolist()
course_texts = catalog_pool["_text"].tolist()
pref_texts = employees_eval["Preferred_Formation"].tolist()

loo_summary = []
ranks_by_model = {}  # all_boosts ranks per model, kept for stratified analysis (section 16)
ablation_summary = []
multilingual_rows = []

random_metrics = random_baseline_metrics()
popularity_metrics = popularity_baseline_metrics()
print("Random baseline:", random_metrics)
print("Popularity baseline:", popularity_metrics)

for model_name in MODELS:
    print("=" * 80)
    print(f"MODEL: {model_name}")
    t0 = time.time()
    sim_matrix, pref_sim_matrix = EMBEDDERS[model_name](emp_texts, course_texts, pref_texts, has_pref)
    embed_elapsed = time.time() - t0

    for ablation_name, overrides in ABLATIONS.items():
        rank_score_matrix_masked = compute_rank_score_matrix(
            sim_matrix, pref_sim_matrix, has_pref, field_score_norm, weight_overrides=overrides,
        )
        ranks = rank_of_held_out(rank_score_matrix_masked)
        metrics = precision_recall_mrr(ranks)

        row = {"model": model_name, "ablation": ablation_name, **metrics}
        if ablation_name == "all_boosts":
            ranks_by_model[model_name] = ranks
            row["embed_elapsed_sec"] = round(embed_elapsed, 1)
            loo_summary.append(row)

            # Multilingual sanity check, computed against the full-boost ranking only.
            hit3 = ranks <= TOP_K
            lang_match_mask = (employees_eval["_held_out_library"].str.upper().map(
                {"EN-US": "EN", "FR-FR": "FR"}).fillna("") == employees_eval["Inferred_Language"])
            multilingual_rows.append({
                "model": model_name,
                "hit3_rate_lang_match": float(hit3[lang_match_mask.values].mean()) if lang_match_mask.any() else None,
                "hit3_rate_lang_mismatch": float(hit3[~lang_match_mask.values].mean()) if (~lang_match_mask).any() else None,
                "n_lang_match": int(lang_match_mask.sum()),
                "n_lang_mismatch": int((~lang_match_mask).sum()),
            })
        else:
            ablation_summary.append(row)

    print(f"  done in {time.time() - t0:.1f}s total (embedding: {embed_elapsed:.1f}s)")

loo_df = pd.DataFrame(loo_summary)
ablation_df = pd.DataFrame(ablation_summary)
multilingual_df = pd.DataFrame(multilingual_rows)

print(loo_df)


## 13. Save leave-one-out + ablation + multilingual results

In [ ]:
baseline_rows = pd.DataFrame([
    {"model": "random_baseline", "ablation": "all_boosts", **random_metrics},
    {"model": "popularity_baseline", "ablation": "all_boosts", **popularity_metrics},
])
loo_df_with_baselines = pd.concat([loo_df, baseline_rows], ignore_index=True)

loo_path = OUTPUTS / "leave_one_out_summary.csv"
ablation_path = OUTPUTS / "ablation_summary.csv"
multilingual_path = OUTPUTS / "multilingual_sanity_check.csv"

loo_df_with_baselines.to_csv(loo_path, index=False)
ablation_df.to_csv(ablation_path, index=False)
multilingual_df.to_csv(multilingual_path, index=False)

print(f"Saved {loo_path}")
print(f"Saved {ablation_path}")
print(f"Saved {multilingual_path}")

print()
print("Leave-one-out summary (incl. baselines):")
print(loo_df_with_baselines)
print()
print("Ablation summary (compare each row's mrr/precision/recall against that model's 'all_boosts' row above):")
print(ablation_df)
print()
print("Multilingual sanity check:")
print(multilingual_df)


## 16. Stratified leave-one-out — popular vs long-tail held-out courses

Section 11 showed the 58 held-out (most-recent-completion) courses skew
heavily toward the most popular courses company-wide — 100% sit in the
top 10% most popular, and one course alone ("Digital transformation for
human resources") accounts for 18 of the 58 cases. That means the single
blended `leave_one_out_summary.csv` number mixes two different questions:
"can we predict which mandatory/generic course someone took" (popularity
wins by construction) and "can we predict which non-obvious,
individually-relevant course someone took" (the question the recommender
is actually built to answer).

This splits the same 58 cases into two strata using `held_out_percentiles`
(already computed in section 11, 1 = most popular course in the pool,
0 = least popular):

- **`popular`** — held-out course is in the top `STRATIFY_THRESHOLD`
  fraction most popular company-wide (default: top 10%)
- **`long_tail`** — everything else

...and reports precision@k / recall@k / MRR separately per stratum, for
every model plus both baselines. Reuses `ranks_by_model` (captured in
section 12, no re-embedding) and recomputes the popularity baseline's
ranks the same way section 10 did.


In [ ]:
STRATIFY_THRESHOLD = 0.10  # top 10% most popular vs everything else ("long-tail")

is_popular = (held_out_percentiles <= STRATIFY_THRESHOLD).values
is_long_tail = ~is_popular
print(f"Popular stratum   (top {int(STRATIFY_THRESHOLD * 100)}% most popular): {is_popular.sum()} employees")
print(f"Long-tail stratum (everything else):        {is_long_tail.sum()} employees")


def precision_recall_mrr_stratum(ranks: np.ndarray, mask: np.ndarray, k: int = TOP_K) -> dict:
    sub = ranks[mask]
    if len(sub) == 0:
        return {"precision_at_k": None, "recall_at_k": None, "mrr": None, "n": 0}
    return precision_recall_mrr(sub, k=k)


def random_baseline_metrics_stratum(mask: np.ndarray, k: int = TOP_K) -> dict:
    N = n_eligible_per_employee[mask]
    if len(N) == 0:
        return {"precision_at_k": None, "recall_at_k": None, "mrr": None, "n": 0}
    hit_rate = np.minimum(k, N) / N
    mrr = harmonic_cumsum[N - 1] / N
    return {
        "precision_at_k": float((hit_rate / k).mean()),
        "recall_at_k": float(hit_rate.mean()),
        "mrr": float(mrr.mean()),
        "n": int(len(N)),
    }


# Recompute popularity-baseline ranks (same logic as section 10's popularity_baseline_metrics,
# just keeping the per-employee ranks array instead of only the aggregated metrics).
pop_matrix = np.tile(popularity_by_col + jitter, (n_eval, 1))
held_out_cols = np.array([course_id_index[cid] for cid in employees_eval["_held_out_course_id"]])
pop_matrix[np.arange(n_eval), held_out_cols] -= 1.0
exclusion_mask = np.zeros((n_eval, n_course), dtype=bool)
for i, excluded in enumerate(employees_eval["Excluded_Course_IDs"]):
    for cid in excluded:
        col = course_id_index.get(cid)
        if col is not None:
            exclusion_mask[i, col] = True
pop_matrix_masked = np.where(exclusion_mask, -np.inf, pop_matrix)
popularity_ranks = rank_of_held_out(pop_matrix_masked)

stratified_rows = []
for stratum_name, mask in [("popular", is_popular), ("long_tail", is_long_tail)]:
    stratified_rows.append({"model": "random_baseline", "stratum": stratum_name,
                             **random_baseline_metrics_stratum(mask)})
    stratified_rows.append({"model": "popularity_baseline", "stratum": stratum_name,
                             **precision_recall_mrr_stratum(popularity_ranks, mask)})
    for model_name in MODELS:
        stratified_rows.append({"model": model_name, "stratum": stratum_name,
                                 **precision_recall_mrr_stratum(ranks_by_model[model_name], mask)})

stratified_df = pd.DataFrame(stratified_rows)
stratified_path = OUTPUTS / "leave_one_out_stratified.csv"
stratified_df.to_csv(stratified_path, index=False)
print(f"\nSaved {stratified_path}\n")
print(stratified_df.to_string(index=False))


## 14. Beyond-accuracy metrics (coverage, novelty, diversity) — no GPU needed

Reads the **existing** `translated_en` recommendation CSVs from
`Recommend_Model_Comparison.ipynb` (`recommendations_translated_en_{model}.csv`)
directly — this section doesn't re-embed anything, so it's runnable
immediately regardless of whether sections 1-12 have finished.

- **Coverage** — fraction of the ~10,765-course active pool that appears
  as *any* employee's top-3 recommendation, for each model.
- **Novelty** — mean self-information of recommended courses
  (`-log2(popularity_fraction)`, Laplace-smoothed since most of the pool
  has zero recorded completions) — higher means the model recommends more
  long-tail, less-obvious courses on average.
- **Diversity** — mean pairwise dissimilarity (`1 - cosine_similarity` on
  each course's `Skills_EN`/`Subjects_EN`/`Description_EN` text) among the
  3 courses in a single employee's recommendation list, averaged across
  all employees.

In [ ]:
def load_comparison_results(model_name: str) -> pd.DataFrame:
    path = COMPARISON_OUTPUTS / f"recommendations_translated_en_{model_name}.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found -- run Recommend_Model_Comparison.ipynb's translated_en "
            f"conditions first, or check COMPARISON_OUTPUTS points at the right folder."
        )
    return pd.read_csv(path)


# Popularity fractions for novelty, reusing the same global_popularity counts as the LOO popularity baseline.
total_completions = global_popularity.sum()
popularity_fraction = {cid: (count + 1) / (total_completions + n_course) for cid, count in global_popularity.items()}
# Laplace smoothing (+1 per course, +n_course in the denominator) so a course with zero
# recorded completions still gets a finite, very small (not undefined) popularity fraction.
default_fraction = 1.0 / (total_completions + n_course)


def novelty(course_ids: pd.Series) -> float:
    fracs = course_ids.astype(str).map(lambda cid: popularity_fraction.get(cid, default_fraction))
    return float((-np.log2(fracs)).mean())


# Course-text similarity matrix for diversity, built once and reused across all 3 models'
# recommendation sets -- only needs the courses that actually appear as a top-3 pick somewhere.
def build_course_similarity_lookup(all_course_ids: set) -> dict:
    ids = sorted(all_course_ids)
    rows = catalog_pool[catalog_pool["Course ID"].astype(str).isin(ids)]
    texts = rows.set_index(rows["Course ID"].astype(str))["_text"]
    texts = texts.reindex(ids).fillna("")
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform(texts.tolist())
    sim = cosine_similarity(tfidf)
    return {cid: {cid2: sim[i, j] for j, cid2 in enumerate(ids)} for i, cid in enumerate(ids)}


def diversity(results_df: pd.DataFrame, sim_lookup: dict) -> float:
    dissimilarities = []
    for email, group in results_df.groupby("Email"):
        ids = group.sort_values("Rank")["Course_ID"].astype(str).tolist()
        ids = [cid for cid in ids if cid in sim_lookup]
        if len(ids) < 2:
            continue
        pair_scores = []
        for i in range(len(ids)):
            for j in range(i + 1, len(ids)):
                pair_scores.append(1.0 - sim_lookup[ids[i]].get(ids[j], 0.0))
        if pair_scores:
            dissimilarities.append(np.mean(pair_scores))
    return float(np.mean(dissimilarities)) if dissimilarities else float("nan")


beyond_accuracy_rows = []
all_recommended_ids = set()
per_model_results = {}
for model_name in MODELS:
    df = load_comparison_results(model_name)
    per_model_results[model_name] = df
    all_recommended_ids.update(df["Course_ID"].dropna().astype(str))

sim_lookup = build_course_similarity_lookup(all_recommended_ids)

for model_name, df in per_model_results.items():
    unique_courses = df["Course_ID"].dropna().astype(str).nunique()
    coverage = unique_courses / n_course
    nov = novelty(df["Course_ID"].dropna())
    div = diversity(df, sim_lookup)
    beyond_accuracy_rows.append({
        "model": model_name,
        "coverage": round(coverage, 4),
        "unique_courses_recommended": unique_courses,
        "novelty": round(nov, 4),
        "diversity": round(div, 4),
    })

beyond_accuracy_df = pd.DataFrame(beyond_accuracy_rows)
beyond_accuracy_path = OUTPUTS / "beyond_accuracy_summary.csv"
beyond_accuracy_df.to_csv(beyond_accuracy_path, index=False)
print(f"Saved {beyond_accuracy_path}")
beyond_accuracy_df


## 15. What to look at once this comes back

- **`leave_one_out_summary.csv`** — the core comparison. Each model's
  `precision_at_k`/`recall_at_k`/`mrr` against `random_baseline` and
  `popularity_baseline`. A model that doesn't clearly beat
  `popularity_baseline` is a meaningful finding in itself, not just a
  disappointing one — it would mean the content-based approach isn't
  adding predictive value over "recommend whatever's popular," which is
  cheap to say honestly in the report if that's what the numbers show.
- **`n` in every row** — the actual eligible-employee count after all three
  drop reasons in section 5. Report this raw number next to every
  percentage, per the small-N honesty concern flagged when this evaluation
  was planned.
- **`ablation_summary.csv`** — compare each `no_X` row's `mrr` against that
  same model's `all_boosts` row in `leave_one_out_summary.csv`. A boost
  that *lowers* MRR when removed is earning its keep; one that raises MRR
  when removed is actively hurting leave-one-out performance despite
  whatever hand-checking motivated it — worth knowing before the weight
  tuning pass.
- **`multilingual_sanity_check.csv`** — if `hit3_rate_lang_mismatch` is
  much lower than `hit3_rate_lang_match`, that's expected and fine (a
  French-history employee's held-out French course should be easier to
  surface than a language-mismatched one). If they're roughly equal, the
  language signal isn't doing much work in practice.
- **`beyond_accuracy_summary.csv`** — coverage/novelty/diversity per model.
  Worth reading alongside the model-comparison hub-course finding: a model
  with low coverage and low novelty is one where a small number of courses
  (like "Creating Your IT Strategy") are doing most of the recommending —
  this gives that a number instead of just the top-10 table.
